# CoalGameRec local run notebook — Mac M4 Pro / 48GB RAM

This notebook runs an end-to-end **local executable prototype** of the CoalGameRec case-study pipeline:

1. load MovieLens-1M (automatic download);
2. optional Amazon Books 2018 loader if you provide `Books_5.json.gz`;
3. convert ratings to implicit positives;
4. build a temporal leave-one-out split;
5. train a small frozen BPR-MF backbone on Apple Silicon (`mps` if available);
6. cache full-catalogue base scores;
7. compute train-only item vectors;
8. compute post-hoc Shapley attributions on validation relevance;
9. apply fixed post-hoc reranking;
10. report HitRate@K and NDCG@K.

**Important:** this is a Mac-local implementation/prototype. It is not the validated official HCCF port required for confirmatory preregistration. The HCCF port, `PORT.md`, validation logs, lockfile/container, ethics determination, and external preregistration remain required real artifacts.

In [1]:
# Auto-reload local package while developing in VS Code/Jupyter.
%load_ext autoreload
%autoreload 2

# If needed, run once in your environment:
# %pip install -r ../requirements.txt

from pathlib import Path
import sys, json, platform

# Robust path setup whether the kernel cwd is code/ or code/notebooks/
CWD = Path.cwd().resolve()
CODE_DIR = CWD if (CWD / 'coalgamerec').exists() else CWD.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import torch

from coalgamerec.data import load_movielens_1m, load_amazon_books_2018, preprocess_temporal_loo, item_user_vectors
from coalgamerec.models import TrainConfig, train_bprmf, cache_full_scores, pick_device
from coalgamerec.metrics import evaluate
from coalgamerec.attribution import compute_shapley_for_users
from coalgamerec.rerank import rerank_all
from coalgamerec.validation import assert_item_vector_isolation, assert_rerank_nonzero, assert_shapley_shapes
import coalgamerec, coalgamerec.rerank as _rerank_mod

print('coalgamerec package:', coalgamerec.__file__)
print('coalgamerec version:', getattr(coalgamerec, '__version__', 'unknown'))
print('rerank module:', _rerank_mod.__file__)
print('rerank sparse fix:', getattr(_rerank_mod, 'SPARSE_FIX_VERSION', 'MISSING - restart/pull required'))
print(platform.platform())
print('torch', torch.__version__)
print('mps available:', torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False)
print('device:', pick_device('auto'))

coalgamerec package: /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/__init__.py
coalgamerec version: 0.2.1-sparse-fix
rerank module: /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/rerank.py
rerank sparse fix: 0.2.1-sparse-safe
macOS-26.5.1-arm64-arm-64bit
torch 2.3.1
mps available: True
device: mps


## Configuration

Defaults are set for a quick Mac M4 Pro feasibility run. For a more complete MovieLens run, increase `SAMPLE_USERS`, `EPOCHS`, and `SHAPLEY_USERS`.

In [2]:
ROOT = CODE_DIR
DATA_RAW = ROOT / 'data' / 'raw'
RESULTS = ROOT / 'results' / 'mac_run'
RESULTS.mkdir(parents=True, exist_ok=True)

# Quick local defaults. Set SAMPLE_USERS=None for full MovieLens-1M.
SAMPLE_USERS = 2000
EPOCHS = 8
DIM = 64
BATCH_SIZE = 4096
SEED = 42

# Shapley is the expensive step. Use None for all users after feasibility testing.
SHAPLEY_USERS = 500
M_PERMUTATIONS = 32  # Mac-feasible prospective run; HCCF/HPC template uses 128.
MAX_PLAYERS_PER_USER = 24  # set None for unbounded; 24 keeps Mac runs feasible
PLAYER_SELECTION = "stratified"
LAMBDA_ATTR = 0.10
KS = (5, 10, 20)

cfg = dict(SAMPLE_USERS=SAMPLE_USERS, EPOCHS=EPOCHS, DIM=DIM, BATCH_SIZE=BATCH_SIZE, SEED=SEED, SHAPLEY_USERS=SHAPLEY_USERS, M_PERMUTATIONS=M_PERMUTATIONS, MAX_PLAYERS_PER_USER=MAX_PLAYERS_PER_USER, PLAYER_SELECTION=PLAYER_SELECTION, LAMBDA_ATTR=LAMBDA_ATTR)
print(json.dumps(cfg, indent=2))

{
  "SAMPLE_USERS": 2000,
  "EPOCHS": 8,
  "DIM": 64,
  "BATCH_SIZE": 4096,
  "SEED": 42,
  "SHAPLEY_USERS": 500,
  "M_PERMUTATIONS": 64,
  "LAMBDA_ATTR": 0.1
}


## Load MovieLens-1M and build temporal leave-one-out split

In [3]:
ratings = load_movielens_1m(DATA_RAW)
print(ratings.head())
print('raw rows:', len(ratings), 'users:', ratings.user_raw.nunique(), 'items:', ratings.item_raw.nunique())

split, stats = preprocess_temporal_loo(ratings, name='ml1m', sample_users=SAMPLE_USERS, sample_seed=SEED)
print(json.dumps(stats, indent=2, default=str))
split.train.head()

   user_raw  item_raw  rating  timestamp  line_idx
0         1      1193       5  978300760         0
1         1       661       3  978302109         1
2         1       914       3  978301968         2
3         1      3408       4  978300275         3
4         1      2355       5  978824291         4
raw rows: 1000209 users: 6040 items: 3706
{
  "name": "ml1m",
  "users": 1991,
  "items": 2599,
  "train_interactions": 193789,
  "val_interactions": 1991,
  "test_interactions": 1991,
  "density_train": 0.037449979312446605,
  "mean_train_per_user": 97.33249623304872,
  "core_log": [
    {
      "iter": 0,
      "users": 1993,
      "items": 2599,
      "edges": 193884
    },
    {
      "iter": 1,
      "users": 1992,
      "items": 2599,
      "edges": 193880
    },
    {
      "iter": 2,
      "users": 1992,
      "items": 2599,
      "edges": 193880
    }
  ],
  "rating_threshold": 4.0,
  "min_uc": 5,
  "min_ic": 5
}


,user,item,timestamp,line_idx,user_raw,item_raw
130,354,141,978298124,130,2,1198
64,354,155,978298151,64,2,1210
88,354,240,978298261,88,2,1293
170,354,1459,978298372,170,2,2943
106,354,171,978298391,106,2,1225


## Optional: Amazon Books 2018 loader

Download `Books_5.json.gz` manually from the UCSD Amazon Reviews 2018 page, then set `AMAZON_BOOKS_5` below. The full file is very large; start with `max_rows` for a feasibility spike.

In [4]:
RUN_AMAZON = False
AMAZON_BOOKS_5 = DATA_RAW / 'Books_5.json.gz'

if RUN_AMAZON:
    amazon = load_amazon_books_2018(AMAZON_BOOKS_5, max_rows=2_000_000)
    amazon_split, amazon_stats = preprocess_temporal_loo(amazon, name='amazon_books_2018_sample', sample_users=50000, sample_seed=SEED)
    print(json.dumps(amazon_stats, indent=2, default=str))


## Train frozen backbone

The notebook uses a small BPR-MF backbone so the full pipeline runs on a Mac. Replace this with the validated HCCF port once that artifact exists.

In [5]:
train_cfg = TrainConfig(dim=DIM, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, device='auto')
model = train_bprmf(split.train, split.n_users, split.n_items, train_cfg, verbose=True)
base_scores = cache_full_scores(model, split.n_users, batch_size=256)
print(base_scores.shape, base_scores.dtype)

train BPR-MF:   0%|          | 0/8 [00:00<?, ?it/s]

cache base scores:   0%|          | 0/8 [00:00<?, ?it/s]

(1991, 2599) float32


## Build train-only item vectors and evaluate base model

In [6]:
X_items = item_user_vectors(split.train_csr)
print('item vectors:', X_items.shape, 'nnz:', X_items.nnz, 'density:', X_items.nnz / (X_items.shape[0] * X_items.shape[1]))

base_summary, base_per_user = evaluate(base_scores, split, X_items, ks=KS)
print('item-vector isolation:', assert_item_vector_isolation(split))
pd.Series(base_summary, name='base').to_frame()

item vectors: (2599, 1991) nnz: 193789 density: 0.037449979312446605
item-vector isolation: {'item_vector_hash': 'db018f8b592b281af96336c9e056e49daa22c749c093de0f1935e11c9fb5238e', 'shape': (2599, 1991), 'nnz': 193789, 'density': 0.037449979312446605}


,base
HitRate@5,0.044701
NDCG@5,0.027703
HitRate@10,0.066801
NDCG@10,0.034815
HitRate@20,0.117529
NDCG@20,0.047594
Coverage@20,0.573682
ILD@20,0.706220


## Compute Shapley attributions using validation relevance

This is the expensive part. The notebook defaults to the first 500 users and `M=64` for speed. For a closer preregistration-like run, set `SHAPLEY_USERS=None` and `M_PERMUTATIONS=128`.

In [7]:
shapley = compute_shapley_for_users(
    split, base_scores, X_items,
    max_users=SHAPLEY_USERS,
    m=M_PERMUTATIONS,
    exact_threshold=8,
    seed=SEED,
    max_players_per_user=MAX_PLAYERS_PER_USER,
    player_selection=PLAYER_SELECTION,
    checkpoint_path=RESULTS / "shapley_notebook_checkpoint.npz",
    save_every=10,
    alpha=1.0, beta=0.0, lambda_pref=0.0, lambda_attr_value=0.10,
    value_mode="pairwise_logsigmoid", n_val_negatives=100, antithetic=True,
)
# Users without Shapley in quick mode get zero weights so the notebook can still evaluate all users.
print('computed users:', len(shapley))
print('shapley shapes:', assert_shapley_shapes(split, shapley))
first_u = next(iter(shapley))
print(first_u, shapley[first_u][:10])

Shapley users:   0%|          | 0/500 [00:00<?, ?it/s]

computed users: 500
shapley shapes: {'checked_users': 500, 'status': 'ok'}
0 [0.0796331  0.13395187 0.1358969  0.15499257 0.1532681  0.12950422
 0.1400622  0.1421482  0.13250057 0.11430774]


## Rerank and evaluate attribution families

In [8]:
print('rerank nonzero:', assert_rerank_nonzero(split, base_scores, X_items, family='uniform'))
families = ['uniform', 'additive-pref', 'attention', 'heuristic-pop', 'shapley-mc']
rows = []
for fam in families:
    scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=LAMBDA_ATTR)
    summary, _ = evaluate(scores, split, X_items, ks=KS)
    summary['family'] = fam
    rows.append(summary)
results = pd.DataFrame(rows).set_index('family')
results.to_csv(RESULTS / 'ml1m_mac_local_results.csv')
results

rerank nonzero: {'inspected_users': 100, 'changed_users': 100, 'max_abs_delta': 2.7667551040649414}


,HitRate@5,NDCG@5,HitRate@10,NDCG@10,HitRate@20,NDCG@20,Coverage@20,ILD@20
family,,,,,,,,
uniform,0.043697,0.027720,0.071321,0.036717,0.116022,0.047855,0.524048,0.691809
additive-pref,0.044199,0.027814,0.071321,0.036630,0.115520,0.047663,0.523663,0.691372
attention,0.045706,0.028551,0.071321,0.036851,0.117027,0.048247,0.522124,0.690093
heuristic-pop,0.043697,0.027620,0.070819,0.036467,0.115520,0.047634,0.522124,0.691410
shapley-mc,0.044701,0.027847,0.068810,0.035585,0.118031,0.047918,0.561755,0.702338


## Reranking strength sensitivity

In [9]:
sens_rows = []
for lam in [0.05, 0.10, 0.20]:
    for fam in ['uniform', 'additive-pref', 'shapley-mc']:
        scores = rerank_all(base_scores, split, X_items, fam, shapley_by_user=shapley, lambda_attr=lam)
        summary, _ = evaluate(scores, split, X_items, ks=KS)
        summary.update({'family': fam, 'lambda_attr': lam})
        sens_rows.append(summary)
sensitivity = pd.DataFrame(sens_rows)
sensitivity.to_csv(RESULTS / 'ml1m_lambda_sensitivity.csv', index=False)
sensitivity

,HitRate@5,NDCG@5,HitRate@10,NDCG@10,HitRate@20,NDCG@20,Coverage@20,ILD@20,family,lambda_attr
0,0.044701,0.028087,0.068810,0.035896,0.115520,0.047630,0.545594,0.698730,uniform,0.05
1,0.044199,0.028002,0.068810,0.035990,0.115520,0.047713,0.545210,0.698467,additive-pref,0.05
2,0.044199,0.027653,0.067303,0.035116,0.117529,0.047747,0.566372,0.704153,shapley-mc,0.05
3,0.043697,0.027720,0.071321,0.036717,0.116022,0.047855,0.524048,0.691809,uniform,0.10
4,0.044199,0.027814,0.071321,0.036630,0.115520,0.047663,0.523663,0.691372,additive-pref,0.10
5,0.044701,0.027847,0.068810,0.035585,0.118031,0.047918,0.561755,0.702338,shapley-mc,0.10
6,0.047212,0.029806,0.074335,0.038407,0.118533,0.049407,0.484032,0.680424,uniform,0.20
7,0.047212,0.029904,0.073832,0.038397,0.119538,0.049772,0.480185,0.679545,additive-pref,0.20
8,0.044701,0.028111,0.069312,0.036020,0.119538,0.048587,0.557907,0.699228,shapley-mc,0.20


## Save run manifest

In [10]:
manifest = {
    'note': 'Mac-local CoalGameRec prototype; not confirmatory HCCF preregistration run',
    'config': cfg,
    'dataset_stats': stats,
    'torch': torch.__version__,
    'device': str(pick_device('auto')),
}
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print('wrote', RESULTS)

wrote /Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/results/mac_run


In [17]:
!python ../scripts/run_q1_pipeline.py --config configs/q1_mac_ml1m.yaml


cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 825.64it/s]
Shapley users: 100%|█| 6015/6015 [1:06:26<00:00,  1.51it/s, hist=165, players=24
cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 986.07it/s]
Shapley users: 100%|█| 6015/6015 [1:40:15<00:00,  1.00s/it, hist=165, players=24
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1384.96it/s]
Shapley users: 100%|█| 6015/6015 [1:19:21<00:00,  1.26it/s, hist=165, players=24
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1176.08it/s]
Shapley users: 100%|█| 6015/6015 [1:06:35<00:00,  1.51it/s, hist=165, players=24
cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 742.78it/s]
Shapley users: 100%|█| 6015/6015 [1:20:32<00:00,  1.24it/s, hist=165, players=24
Python(20993) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(20994) MallocStackLogging: can't turn off malloc stack logging because it was not ena

In [18]:
!python ../scripts/analyze_q1_results.py  --run-dir results/journal_runs/ml1m_mac_journal_v1

                           contrast  ... holm_threshold
0     shapley-mc_vs_uniform_NDCG@20  ...          0.025
1  shapley-mc_vs_uniform_HitRate@20  ...          0.050

[2 rows x 14 columns]


In [20]:
!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_mac_ml1m.yaml

cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 285.22it/s]
shapley-mc users:   2%| | 115/6015 [03:30<3:00:21,  1.83s/it, hist=257, players=
Traceback (most recent call last):
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/notebooks/../scripts/run_q1_pipeline.py", line 26, in <module>
    main()
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/notebooks/../scripts/run_q1_pipeline.py", line 21, in main
    out = run_pipeline(args.config)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/pipeline.py", line 196, in run_pipeline
    summary_df, per_user_df = run_seed(split, item_vectors, cfg, int(seed), out_dir)
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/pipeline.py", line 88, i

In [22]:
!python ../scripts/run_q1_pipeline.py --config configs/q1_mac_ml1m.yaml


Python(33208) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 351.40it/s]
shapley-mc users: 100%|█| 6015/6015 [1:05:02<00:00,  1.54it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:46<00:00, 14.81it/s, hist=165, players
cache base scores: 100%|███████████████████████| 24/24 [00:00<00:00, 785.19it/s]
shapley-mc users: 100%|█| 6015/6015 [50:25<00:00,  1.99it/s, hist=165, players=2
loo-marginal users: 100%|█| 6015/6015 [1:22:43<00:00,  1.21it/s, hist=165, playe
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1101.47it/s]
shapley-mc users: 100%|█| 6015/6015 [1:01:02<00:00,  1.64it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [07:16<00:00, 13.78it/s, hist=165, players
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1066.00it/s]
shapley-mc users: 100%|█| 6015/6015 [49:10<00:00,  2.04it/s, hist=165, players=2
loo-marginal users: 100%|█| 6015/6015 [06:42<00:00, 14.96it/s, hist=165, players
cache base scores: 100%|████

In [24]:
!python ../scripts/analyze_q1_results.py \
  --run-dir results/journal_runs/ml1m_mac_journal_v2_prospective

Python(10715) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


                           contrast  ... holm_threshold
0     shapley-mc_vs_uniform_NDCG@20  ...          0.025
1  shapley-mc_vs_uniform_HitRate@20  ...          0.050

[2 rows x 14 columns]


In [25]:
!python ../scripts/analyze_q1_results.py \
  --run-dir results/journal_runs/ml1m_mac_journal_v2_prospective \
  --treatment shapley-mc \
  --controls uniform additive-pref attention loo-marginal \
  --output-prefix paired_bootstrap_all_controls

Python(11512) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


                                 contrast  ... holm_threshold
0           shapley-mc_vs_uniform_NDCG@20  ...       0.006250
1        shapley-mc_vs_uniform_HitRate@20  ...       0.007143
2     shapley-mc_vs_additive-pref_NDCG@20  ...       0.008333
3  shapley-mc_vs_additive-pref_HitRate@20  ...       0.010000
4         shapley-mc_vs_attention_NDCG@20  ...       0.012500
5      shapley-mc_vs_attention_HitRate@20  ...       0.016667
6      shapley-mc_vs_loo-marginal_NDCG@20  ...       0.050000
7   shapley-mc_vs_loo-marginal_HitRate@20  ...       0.025000

[8 rows x 18 columns]


In [27]:
!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_lightgcn_ml1m.yaml

Python(12906) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


cache base scores: 100%|████████████████████████| 24/24 [00:02<00:00,  9.74it/s]
shapley-mc users: 100%|█| 6015/6015 [1:51:15<00:00,  1.11s/it, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:44<00:00, 14.86it/s, hist=165, players
cache base scores: 100%|████████████████████████| 24/24 [00:01<00:00, 12.12it/s]
shapley-mc users: 100%|█| 6015/6015 [2:17:04<00:00,  1.37s/it, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:26<00:00, 15.56it/s, hist=165, players
cache base scores: 100%|████████████████████████| 24/24 [00:01<00:00, 12.59it/s]
shapley-mc users: 100%|█| 6015/6015 [1:50:40<00:00,  1.10s/it, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [10:50<00:00,  9.25it/s, hist=165, players
cache base scores: 100%|████████████████████████| 24/24 [00:02<00:00, 11.52it/s]
shapley-mc users: 100%|█| 6015/6015 [1:31:02<00:00,  1.10it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:27<00:00, 15.54it/s, hist=165, players
cache base scores: 100%|████

In [31]:
!python ../scripts/analyze_q1_results.py \
  --run-dir results/journal_runs/ml1m_lightgcn_v3_prospective \
  --treatment shapley-mc \
  --controls uniform additive-pref attention loo-marginal \
  --output-prefix paired_bootstrap_all_controls

Python(98570) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


                                 contrast  ... holm_threshold
0           shapley-mc_vs_uniform_NDCG@20  ...       0.006250
1        shapley-mc_vs_uniform_HitRate@20  ...       0.007143
2     shapley-mc_vs_additive-pref_NDCG@20  ...       0.008333
3  shapley-mc_vs_additive-pref_HitRate@20  ...       0.010000
4         shapley-mc_vs_attention_NDCG@20  ...       0.012500
5      shapley-mc_vs_attention_HitRate@20  ...       0.016667
6      shapley-mc_vs_loo-marginal_NDCG@20  ...       0.025000
7   shapley-mc_vs_loo-marginal_HitRate@20  ...       0.050000

[8 rows x 18 columns]


In [32]:
!python ../scripts/cost_effectiveness.py \
  --run-dir results/journal_runs/ml1m_lightgcn_v3_prospective

Python(98596) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


          family  ...  delta_NDCG_vs_uniform_per_attribution_hour
0  additive-pref  ...                                    1.019561
1      attention  ...                                    1.695183
2  heuristic-pop  ...                                    0.771441
3   loo-marginal  ...                                    0.006124
4     shapley-mc  ...                                    0.000318
5        uniform  ...                                    0.000000

[6 rows x 9 columns]


In [33]:
!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_lightgcn_amazon_template.yaml

Python(98650) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Traceback (most recent call last):
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/notebooks/../scripts/run_q1_pipeline.py", line 26, in <module>
    main()
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/notebooks/../scripts/run_q1_pipeline.py", line 21, in main
    out = run_pipeline(args.config)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/pipeline.py", line 204, in run_pipeline
    split, dataset_stats = prepare_split(cfg)
                           ^^^^^^^^^^^^^^^^^^
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coalgamerec/pipeline.py", line 36, in prepare_split
    raw = load_amazon_books_2018(path, max_rows=ds.get("max_rows"))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/coa

In [34]:
!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_lightgcn_ml1m.yaml

Python(99517) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


loo-marginal users: 100%|██████████████| 6015/6015 [00:00<00:00, 1400818.35it/s]
Wrote run artifacts to: results/journal_runs/ml1m_lightgcn_v3_prospective


In [3]:

!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_lightgcn_ml1m.yaml


cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 3149.17it/s]
shapley-mc users: 100%|█| 6015/6015 [1:28:36<00:00,  1.13it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:41<00:00, 14.98it/s, hist=165, players
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 3800.48it/s]
shapley-mc users: 100%|█| 6015/6015 [1:28:38<00:00,  1.13it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:36<00:00, 15.17it/s, hist=165, players
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1705.15it/s]
shapley-mc users: 100%|█| 6015/6015 [1:30:35<00:00,  1.11it/s, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:38<00:00, 15.11it/s, hist=165, players
cache base scores: 100%|██████████████████████| 24/24 [00:00<00:00, 1091.24it/s]
shapley-mc users: 100%|█| 6015/6015 [2:49:19<00:00,  1.69s/it, hist=165, players
loo-marginal users: 100%|█| 6015/6015 [06:44<00:00, 14.88it/s, hist=165, players
cache base scores: 100%|████

In [4]:
!python ../scripts/analyze_q1_results.py \
  --run-dir results/journal_runs/ml1m_lightgcn_v3_prospective \
  --treatment shapley-mc \
  --controls uniform additive-pref attention loo-marginal \
  --output-prefix paired_bootstrap_all_controls

                                 contrast  ... holm_threshold
0           shapley-mc_vs_uniform_NDCG@20  ...       0.006250
1        shapley-mc_vs_uniform_HitRate@20  ...       0.007143
2     shapley-mc_vs_additive-pref_NDCG@20  ...       0.008333
3  shapley-mc_vs_additive-pref_HitRate@20  ...       0.010000
4         shapley-mc_vs_attention_NDCG@20  ...       0.012500
5      shapley-mc_vs_attention_HitRate@20  ...       0.016667
6      shapley-mc_vs_loo-marginal_NDCG@20  ...       0.025000
7   shapley-mc_vs_loo-marginal_HitRate@20  ...       0.050000

[8 rows x 18 columns]


In [5]:
!python ../scripts/analyze_q1_results.py \
  --run-dir results/journal_runs/ml1m_lightgcn_v3_prospective \
  --treatment shapley-mc \
  --controls uniform additive-pref attention loo-marginal \
  --output-prefix paired_bootstrap_all_controls

                                 contrast  ... holm_threshold
0           shapley-mc_vs_uniform_NDCG@20  ...       0.006250
1        shapley-mc_vs_uniform_HitRate@20  ...       0.007143
2     shapley-mc_vs_additive-pref_NDCG@20  ...       0.008333
3  shapley-mc_vs_additive-pref_HitRate@20  ...       0.010000
4         shapley-mc_vs_attention_NDCG@20  ...       0.012500
5      shapley-mc_vs_attention_HitRate@20  ...       0.016667
6      shapley-mc_vs_loo-marginal_NDCG@20  ...       0.025000
7   shapley-mc_vs_loo-marginal_HitRate@20  ...       0.050000

[8 rows x 18 columns]


In [12]:
!python scripts/prepare_amazon_books.py \
  --dest ../data/raw/Books_5.json.gz \
  --retries 20

/opt/homebrew/Cellar/python@3.12/3.12.13_4/Frameworks/Python.framework/Versions/3.12/Resources/Python.app/Contents/MacOS/Python: can't open file '/Users/mlouhichi/Desktop/CoalGameRec/next-paper/paper-ideas/CoalGameRec/code/notebooks/scripts/prepare_amazon_books.py': [Errno 2] No such file or directory


In [13]:
!python ../scripts/run_q1_pipeline.py \
  --config configs/q1_lightgcn_amazon_template.yaml

cache base scores: 100%|███████████████████████| 58/58 [00:00<00:00, 921.47it/s]
shapley-mc users: 100%|█| 7417/7417 [27:40<00:00,  4.47it/s, hist=6, players=6, 
loo-marginal users: 100%|█| 7417/7417 [02:07<00:00, 58.17it/s, hist=6, players=6
cache base scores: 100%|██████████████████████| 58/58 [00:00<00:00, 1043.54it/s]
shapley-mc users: 100%|█| 7417/7417 [27:36<00:00,  4.48it/s, hist=6, players=6, 
loo-marginal users: 100%|█| 7417/7417 [02:06<00:00, 58.53it/s, hist=6, players=6
cache base scores: 100%|██████████████████████| 58/58 [00:00<00:00, 1250.78it/s]
shapley-mc users: 100%|█| 7417/7417 [27:32<00:00,  4.49it/s, hist=6, players=6, 
loo-marginal users: 100%|█| 7417/7417 [02:07<00:00, 58.07it/s, hist=6, players=6
cache base scores: 100%|██████████████████████| 58/58 [00:00<00:00, 1387.83it/s]
shapley-mc users: 100%|█| 7417/7417 [27:35<00:00,  4.48it/s, hist=6, players=6, 
loo-marginal users: 100%|█| 7417/7417 [02:06<00:00, 58.65it/s, hist=6, players=6
cache base scores: 100%|████